In [1]:
import pandas as pd
import numpy as np

def process_buy_hold_trades(file_path: str):
    """
    End-to-end Buy-Hold strategy PnL computation.

    Logic
    -----
    1. Identify trades using:
       - Month gaps
       - Quantity changes
    2. Compute per-trade returns
    3. Compute compounded strategy return

    Returns
    -------
    bh_pivot : pd.DataFrame
        Trade-level details with buy, sell, PnL, dates
    strategy_return : float
        Compounded strategy return
    """

    # ----------------------------
    # Step 1: Read & prepare data
    # ----------------------------
    df = pd.read_excel(file_path)

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Ticker", "Date"])

    df["YearMonth"] = df["Date"].dt.to_period("M")

    df = df[["Ticker", "Date", "YearMonth", "Buy_Hold_Value", "Quantity"]]

    # ----------------------------
    # Step 2: Monthly snapshot
    # ----------------------------
    df_month = (
        df.drop_duplicates(subset=["Ticker", "YearMonth"])
          .sort_values(["Ticker", "YearMonth"])
    )

    # ----------------------------
    # Step 3: Trade ID logic
    # ----------------------------
    def get_trade_id(g):
        diff_months = g["YearMonth"].diff().apply(
            lambda x: x.n if pd.notna(x) else 1
        )

        new_trade = (
            diff_months.isna() |                # first row
            (diff_months > 1) |                 # month gap
            (g["Quantity"] != g["Quantity"].shift(1))  # quantity change
        )

        return new_trade.cumsum()

    df_month["trade_id"] = (
        df_month.groupby("Ticker", group_keys=False)
                .apply(get_trade_id)
    )

    # ----------------------------
    # Step 4: Merge trade_id back
    # ----------------------------
    df = df.merge(
        df_month[["Ticker", "YearMonth", "trade_id"]],
        on=["Ticker", "YearMonth"],
        how="left"
    )

    # ----------------------------
    # Step 5: Identify buy / sell
    # ----------------------------
    first_date = df.groupby(["Ticker", "trade_id"])["Date"].transform("min")
    last_date  = df.groupby(["Ticker", "trade_id"])["Date"].transform("max")

    df["Transaction"] = np.select(
        [
            df["Date"] == first_date,
            df["Date"] == last_date
        ],
        ["buy", "sell"],
        default=""
    )

    # ----------------------------
    # Step 6: Pivot to trade level
    # ----------------------------
    trades = df[df["Transaction"] != ""]

    bh_pivot = (
        trades.pivot_table(
            index=["Ticker", "trade_id"],
            columns="Transaction",
            values="Buy_Hold_Value",
            aggfunc="first"
        )
        .reset_index()
    )

    # ----------------------------
    # Step 7: Per-trade return
    # ----------------------------
    bh_pivot["Profit/Loss%"] = bh_pivot["sell"] / bh_pivot["buy"] - 1
    bh_pivot["Profit/Loss"] = bh_pivot["sell"] - bh_pivot["buy"]

    # ----------------------------
    # Step 8: Add Buy/Sell dates
    # ----------------------------
    buy_dates = trades[trades["Transaction"] == "buy"][
        ["Ticker", "trade_id", "Date"]
    ].rename(columns={"Date": "Buy Date"})

    sell_dates = trades[trades["Transaction"] == "sell"][
        ["Ticker", "trade_id", "Date"]
    ].rename(columns={"Date": "Sell Date"})

    bh_pivot = (
        bh_pivot.merge(buy_dates, on=["Ticker", "trade_id"])
                 .merge(sell_dates, on=["Ticker", "trade_id"])
    )

    # ----------------------------
    # Step 9: Strategy return (CRITICAL)
    # ----------------------------
    strategy_return = (1 + bh_pivot["Profit/Loss"]).prod() - 1

    return bh_pivot, strategy_return

#Absolute logic, if a stock remains for more than a month inside the portfolio without quantity change, the return shall be calculated for the whole period, eg 3 months

In [2]:
result = process_buy_hold_trades('Momentum_Maxfolio.xlsx')[0]
result

C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: RuntimeWarning: overflow encountered in scalar multiply
  new_data = np.array([self.freq.base * x for x in new_i8_data])
C:\Users\anike\AppData\Roaming\Python\Python314\site-packages\pandas\core\arrays\datetimelike.py:1329: R

,Ticker,trade_id,buy,sell,Profit/Loss%,Profit/Loss,Buy Date,Sell Date
0,ABCAPITAL,1,3.729577,3.746334,0.004493,0.016757,2025-12-01,2025-12-31
1,ANANDRATHI,1,3.754172,3.500045,-0.067692,-0.254127,2026-01-01,2026-01-30
2,APLAPOLLO,1,3.286094,2.915978,-0.112631,-0.370116,2026-03-02,2026-03-12
3,ASHOKLEY,1,3.329454,3.140845,-0.056649,-0.188609,2026-02-01,2026-03-12
4,AUBANK,1,3.731353,3.904083,0.046291,0.172730,2025-12-01,2025-12-31
5,AXISBANK,1,3.387995,3.497945,0.032453,0.109951,2026-02-01,2026-02-27
6,BANKBARODA,1,3.235005,2.958408,-0.085501,-0.276597,2026-03-02,2026-03-12
7,BANKINDIA,1,3.175487,3.230368,0.017283,0.054881,2026-02-01,2026-03-12
8,BHARATFORG,1,3.249130,3.057567,-0.058958,-0.191563,2026-03-02,2026-03-12
9,CANBK,1,3.759071,3.409827,-0.092907,-0.349245,2026-01-01,2026-03-12


In [3]:
import numpy as np

mapping = {
    "SILVERBEES": "SILVER",
    "SILVER": "SILVER",
    "GOLDBEES": "GOLD",
    "MOGSEC": "DEBT"
}

result["Asset_Class"] = result["Ticker"].map(mapping).fillna("EQUITIES")
result


,Ticker,trade_id,buy,sell,Profit/Loss%,Profit/Loss,Buy Date,Sell Date,Asset_Class
0,ABCAPITAL,1,3.729577,3.746334,0.004493,0.016757,2025-12-01,2025-12-31,EQUITIES
1,ANANDRATHI,1,3.754172,3.500045,-0.067692,-0.254127,2026-01-01,2026-01-30,EQUITIES
2,APLAPOLLO,1,3.286094,2.915978,-0.112631,-0.370116,2026-03-02,2026-03-12,EQUITIES
3,ASHOKLEY,1,3.329454,3.140845,-0.056649,-0.188609,2026-02-01,2026-03-12,EQUITIES
4,AUBANK,1,3.731353,3.904083,0.046291,0.172730,2025-12-01,2025-12-31,EQUITIES
5,AXISBANK,1,3.387995,3.497945,0.032453,0.109951,2026-02-01,2026-02-27,EQUITIES
6,BANKBARODA,1,3.235005,2.958408,-0.085501,-0.276597,2026-03-02,2026-03-12,EQUITIES
7,BANKINDIA,1,3.175487,3.230368,0.017283,0.054881,2026-02-01,2026-03-12,EQUITIES
8,BHARATFORG,1,3.249130,3.057567,-0.058958,-0.191563,2026-03-02,2026-03-12,EQUITIES
9,CANBK,1,3.759071,3.409827,-0.092907,-0.349245,2026-01-01,2026-03-12,EQUITIES


In [4]:
(result.groupby("Asset_Class")["Profit/Loss"].sum() / 100).reset_index()

#copy paste values from here to google sheets

,Asset_Class,Profit/Loss
0,DEBT,0.139995
1,EQUITIES,-0.024568
2,GOLD,-0.018486
3,SILVER,0.036180


In [5]:
result.to_excel('PnL_Momentum_Maxfolio.xlsx', index=False)
